In [1]:

# Cell 1

import numpy as np
import pandas as pd
import torch
import joblib
import torch

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [2]:
import pandas as pd

data_path = "/kaggle/input/datasets/joelleiliovits/new-data-csv/new_data.csv"
df = pd.read_csv(data_path)

In [3]:
df = df[["text", "tags"]].copy()
df = df.dropna(subset=["text", "tags"])

df["text"] = df["text"].astype(str).str.strip()
df["tags"] = df["tags"].astype(str).str.strip()

df = df[(df["text"] != "") & (df["tags"] != "")]



In [4]:
df["label_list"] = df["tags"].apply(lambda x: x.split())
df[["text", "tags", "label_list"]].head()  


,text,tags,label_list
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]"
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]"
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]"
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]"
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]"


In [5]:
df["num_labels"] = df["label_list"].apply(len)

print(df["num_labels"].value_counts().sort_index())
df[["text", "tags", "label_list", "num_labels"]].head()

num_labels
1    1000000
2     701000
3     112707
4       7949
5        304
Name: count, dtype: int64


,text,tags,label_list,num_labels
0,loop through elements of list in a pandas data...,python pandas,"[python, pandas]",2
1,ssl error when running pip search in python 2....,python ssl,"[python, ssl]",2
2,how do i clear errno in c# how do i clear errn...,c# linux,"[c#, linux]",2
3,segmentation fault as soon the binary launch h...,linux debugging,"[linux, debugging]",2
4,changing data in jsp using ajax i have jsp pag...,javascript java,"[javascript, java]",2


In [6]:
df_1 = df[df["num_labels"] == 1].copy()
df_2 = df[df["num_labels"] == 2].copy()
df_3_plus = df[df["num_labels"] >= 3].copy()



In [7]:
RANDOM_STATE = 42

df_1_sample = df_1.sample(
    n=min(850000, len(df_1)),
    random_state=RANDOM_STATE
).copy()

df_2_sample = df_2.sample(
    n=min(320000, len(df_2)),
    random_state=RANDOM_STATE
).copy()

df_final = pd.concat([df_1_sample, df_2_sample, df_3_plus], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Final shape:", df_final.shape)
print(df_final["num_labels"].value_counts().sort_index())

Final shape: (1290960, 4)
num_labels
1    850000
2    320000
3    112707
4      7949
5       304
Name: count, dtype: int64


In [8]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df_final["label_list"])

print("Number of labels:", len(mlb.classes_))
print("y shape:", y.shape)
print("First labels:", mlb.classes_[:20])

Number of labels: 52
y shape: (1290960, 52)
First labels: ['apache' 'asp.net-core' 'authentication' 'azure' 'bash' 'c#'
 'computer-vision' 'cors' 'cuda' 'debugging' 'deep-learning' 'django'
 'dns' 'docker' 'fastapi' 'firewall' 'gpu' 'http' 'inference' 'java']


In [9]:
X = df_final["text"].tolist()



In [10]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True
)



In [11]:
from datasets import Dataset
import numpy as np

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

train_dataset = Dataset.from_dict({
    "text": X_train if isinstance(X_train, list) else X_train.tolist(),
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val if isinstance(X_val, list) else X_val.tolist(),
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test if isinstance(X_test, list) else X_test.tolist(),
    "labels": y_test.tolist()
})

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 1032768
Val: 129096
Test: 129096


In [12]:
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
# df فيه عمود text = title + body
sample_df = df.sample(n=min(100_000, len(df)), random_state=42).copy()

texts = sample_df["text"].fillna("").astype(str).tolist()

lengths = []
batch_size = 512

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    enc = tokenizer(
        batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    lengths.extend(len(x) for x in enc["input_ids"])

lengths = np.array(lengths)

print("count:", len(lengths))
print("min:", lengths.min())
print("median:", int(np.percentile(lengths, 50)))
print("p90:", int(np.percentile(lengths, 90)))
print("p95:", int(np.percentile(lengths, 95)))
print("p99:", int(np.percentile(lengths, 99)))
print("max:", lengths.max())

print("over_64 :", round((lengths > 64).mean() * 100, 2), "%")
print("over_128:", round((lengths > 128).mean() * 100, 2), "%")
print("over_256:", round((lengths > 256).mean() * 100, 2), "%")

Token indices sequence length is longer than the specified maximum sequence length for this model (709 > 512). Running this sequence through the model will result in indexing errors


count: 100000
min: 14
median: 191
p90: 462
p95: 575
p99: 809
max: 1693
over_64 : 94.79 %
over_128: 71.85 %
over_256: 33.62 %


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=384
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/1032768 [00:00<?, ? examples/s]

Map:   0%|          | 0/129096 [00:00<?, ? examples/s]

Map:   0%|          | 0/129096 [00:00<?, ? examples/s]

In [15]:
num_labels = y_train.shape[1]

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification"
)

print("Num labels:", num_labels)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Num labels: 52


In [16]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/unixcode_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    save_total_limit=3,
    save_only_model=False,

    per_device_train_batch_size=20,
    per_device_eval_batch_size=20,

    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1_at_3",
    greater_is_better=True,

    fp16=True,
    bf16=False,

    report_to="none"
)

In [17]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

THRESHOLD = 0.35

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def top_k_binary_predictions(y_scores, k):
    y_pred = np.zeros_like(y_scores, dtype=int)
    topk_idx = np.argsort(-y_scores, axis=1)[:, :k]

    for i in range(y_scores.shape[0]):
        y_pred[i, topk_idx[i]] = 1

    return y_pred

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    results = {}

    probs = sigmoid(logits)
    preds_thr = (probs >= THRESHOLD).astype(int)

    results["precision_threshold"] = precision_score(labels, preds_thr, average="micro", zero_division=0)
    results["recall_threshold"] = recall_score(labels, preds_thr, average="micro", zero_division=0)
    results["f1_threshold"] = f1_score(labels, preds_thr, average="micro", zero_division=0)

    for k in [1, 2, 3, 4, 5]:
        preds_k = top_k_binary_predictions(logits, k)

        results[f"precision_at_{k}"] = precision_score(labels, preds_k, average="micro", zero_division=0)
        results[f"recall_at_{k}"] = recall_score(labels, preds_k, average="micro", zero_division=0)
        results[f"f1_at_{k}"] = f1_score(labels, preds_k, average="micro", zero_division=0)

    return results

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [19]:
from pathlib import Path
import shutil
import re
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = Path("/kaggle/working/deberta_pair_cls")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_step(path):
    match = re.search(r"checkpoint-(\d+)", path.name)
    return int(match.group(1)) if match else -1

input_checkpoints = [
    p for p in Path("/kaggle/input").rglob("checkpoint-*")
    if p.is_dir()
]

input_checkpoints = sorted(input_checkpoints, key=checkpoint_step)

if len(input_checkpoints) == 0:
    print("No checkpoints found in /kaggle/input.")
    print("Training will start from scratch.")
else:
    print(f"Found {len(input_checkpoints)} checkpoint(s) in /kaggle/input:")

    for p in input_checkpoints:
        print(" -", p)

    print("\nCopying checkpoints to:", OUTPUT_DIR)

    for ckpt in input_checkpoints:
        dst = OUTPUT_DIR / ckpt.name

        if dst.exists():
            print(f"Already exists, skipping: {dst}")
            continue

        shutil.copytree(ckpt, dst)
        print(f"Copied: {ckpt.name}")

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
if last_checkpoint is None:
    print("\nNo checkpoint available in OUTPUT_DIR.")
    print("Next training cell should start from scratch.")
else:
    print("\nLast checkpoint available:")
    print(last_checkpoint)
    print("Next training cell should resume from this checkpoint.")

Found 3 checkpoint(s) in /kaggle/input:
 - /kaggle/input/notebooks/mikeelio4/modernbert-300-6/deberta_pair_cls/checkpoint-5164
 - /kaggle/input/notebooks/mikeelio4/modernbert-300-6/deberta_pair_cls/checkpoint-10328
 - /kaggle/input/notebooks/mikeelio4/modernbert-300-6/unixcode_results/checkpoint-15492

Copying checkpoints to: /kaggle/working/deberta_pair_cls
Copied: checkpoint-5164
Copied: checkpoint-10328
Copied: checkpoint-15492

Last checkpoint available:
/kaggle/working/deberta_pair_cls/checkpoint-15492
Next training cell should resume from this checkpoint.


In [ ]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
1,0.087583,0.047940,0.830583,0.870259,0.849957,0.922090,0.641431,0.756506,0.627201,0.870986,0.729206,0.452115,0.940442,0.610411,0.348155,0.964263,0.511153,0.281958,0.974803,0.436790
2,0.046111,0.043313,0.830411,0.886388,0.857485,0.930782,0.647471,0.763633,0.631698,0.877236,0.734437,0.455835,0.948199,0.615440,0.350570,0.970976,0.514705,0.283717,0.980919,0.439523


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Epoch,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
3,0.042246,0.042061,0.823606,0.893889,0.857310,0.931795,0.648565,0.764040,0.633890,0.879283,0.736987,0.457181,0.948004,0.619258,0.366424,0.971350,0.518961,0.282240,0.980733,0.438334


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [20]:
if last_checkpoint is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=last_checkpoint)

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Epoch,Training Loss,Validation Loss,Precision Threshold,Recall Threshold,F1 Threshold,Precision At 1,Recall At 1,F1 At 1,Precision At 2,Recall At 2,F1 At 2,Precision At 3,Recall At 3,F1 At 3,Precision At 4,Recall At 4,F1 At 4,Precision At 5,Recall At 5,F1 At 5
4,0.040000,0.041210,0.826720,0.894697,0.859366,0.933143,0.659502,0.766209,0.635041,0.888882,0.739325,0.457963,0.950635,0.633316,0.349891,0.972647,0.526647,0.282545,0.981794,0.439808
5,0.038492,0.040906,0.828289,0.895187,0.860440,0.934305,0.662309,0.768162,0.633525,0.880555,0.741888,0.456095,0.950910,0.635494,0.350009,0.972976,0.528821,0.282692,0.982305,0.439036


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [21]:
val_results = trainer.evaluate(eval_dataset=val_dataset)
print("Validation Results:", val_results)

test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Test Results:", test_results)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation Results: {'eval_loss': 0.040906041860580444, 'eval_precision_threshold': 0.8297892180791186, 'eval_recall_threshold': 0.896686774261274, 'eval_f1_threshold': 0.8619396656326564, 'eval_precision_at_1': 0.9358047034764826, 'eval_recall_at_1': 0.65080905097465, 'eval_f1_at_1': 0.7676622016483778, 'eval_precision_at_2': 0.6350246638160749, 'eval_recall_at_2': 0.8820549125479788, 'eval_f1_at_2': 0.738387629490642, 'eval_precision_at_3': 0.4575946892235236, 'eval_recall_at_3': 0.9524095117867775, 'eval_f1_at_3': 0.6179939350524477, 'eval_precision_at_4': 0.3515089080993989, 'eval_recall_at_4': 0.9744757373801539, 'eval_f1_at_4': 0.5163210549702839, 'eval_precision_at_5': 0.28419195017661275, 'eval_recall_at_5': 0.9838050296351724, 'eval_f1_at_5': 0.4405361857420068, 'eval_runtime': 821.0318, 'eval_samples_per_second': 157.236, 'eval_steps_per_second': 3.932, 'epoch': 5.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Test Results: {'eval_loss': 0.04075715318322182, 'eval_precision_threshold': 0.8307930853210689, 'eval_recall_threshold': 0.8962224118744389, 'eval_f1_threshold': 0.8622661713670992, 'eval_precision_at_1': 0.9355258412344302, 'eval_recall_at_1': 0.649728887228312, 'eval_f1_at_1': 0.7668161287046705, 'eval_precision_at_2': 0.6352919067980418, 'eval_recall_at_2': 0.8812234601882664, 'eval_f1_at_2': 0.7382769385756576, 'eval_precision_at_3': 0.45808269814711533, 'eval_recall_at_3': 0.9521271067075958, 'eval_f1_at_3': 0.618380138007783, 'eval_precision_at_4': 0.3522176829646155, 'eval_recall_at_4': 0.9751147473563675, 'eval_f1_at_4': 0.5171770316501921, 'eval_precision_at_5': 0.28472798537522464, 'eval_recall_at_5': 0.9843237811335767, 'eval_f1_at_5': 0.4412343092485445, 'eval_runtime': 821.8081, 'eval_samples_per_second': 157.088, 'eval_steps_per_second': 3.928, 'epoch': 5.0}
